# OMOP CDM Vocabulary Tables

Creates and loads the standardized OMOP vocabulary tables required for concept mapping.

## Vocabulary Sources

**Option 1: Pre-built vocabularies (recommended for quick start)**
```
s3://hls-eng-data-public/data/rwe/omop-vocabs/
```

**Option 2: Download from OHDSI Athena**
1. Visit [OHDSI Athena](https://athena.ohdsi.org/)
2. Register and download vocabularies (SNOMED, LOINC, RxNorm, etc.)
3. Upload to your Databricks Volume or S3 path

## Tables

| Table | Records | Description |
|-------|---------|-------------|
| concept | ~6M | Master concept lookup |
| vocabulary | ~60 | Vocabulary definitions |
| domain | ~50 | Domain definitions |
| concept_class | ~400 | Concept class definitions |
| concept_relationship | ~30M | Relationships between concepts |
| relationship | ~700 | Relationship type definitions |
| concept_synonym | ~3M | Concept synonyms |
| concept_ancestor | ~80M | Hierarchical relationships |
| drug_strength | ~2M | Drug strength information |

## Pediatric Focus

This vocabulary setup includes pediatric-specific concepts for:
- Growth charts (WHO, CDC percentiles)
- Developmental milestones
- Pediatric vital sign ranges
- Age-appropriate diagnoses and procedures
- Immunization schedules

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';
DECLARE OR REPLACE VARIABLE vocab_path STRING DEFAULT 's3://hls-eng-data-public/data/rwe/omop-vocabs/';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);
SET VARIABLE vocab_path = COALESCE(:vocab_path, vocab_path);

USE IDENTIFIER(catalog_use || '.' || gold_schema);
SELECT current_catalog(), current_schema(), vocab_path AS vocabulary_source;

## Create Vocabulary Tables

In [ ]:
-- CONCEPT table
CREATE TABLE IF NOT EXISTS concept (
  concept_id BIGINT NOT NULL COMMENT 'Unique identifier for each concept'
  ,concept_name STRING NOT NULL COMMENT 'Concept name'
  ,domain_id STRING NOT NULL COMMENT 'Domain the concept belongs to'
  ,vocabulary_id STRING NOT NULL COMMENT 'Vocabulary the concept belongs to'
  ,concept_class_id STRING NOT NULL COMMENT 'Concept class'
  ,standard_concept STRING COMMENT 'S=Standard, C=Classification, NULL=Non-standard'
  ,concept_code STRING NOT NULL COMMENT 'Code in source vocabulary'
  ,valid_start_date DATE NOT NULL COMMENT 'Concept validity start'
  ,valid_end_date DATE NOT NULL COMMENT 'Concept validity end'
  ,invalid_reason STRING COMMENT 'Reason if concept is invalid'
)
USING DELTA
COMMENT 'OMOP CDM Concept table - Master vocabulary lookup'
TBLPROPERTIES ('quality' = 'gold');

In [ ]:
-- VOCABULARY table
CREATE TABLE IF NOT EXISTS vocabulary (
  vocabulary_id STRING NOT NULL
  ,vocabulary_name STRING NOT NULL
  ,vocabulary_reference STRING
  ,vocabulary_version STRING
  ,vocabulary_concept_id BIGINT NOT NULL
)
USING DELTA
COMMENT 'OMOP CDM Vocabulary definitions';

In [ ]:
-- DOMAIN table
CREATE TABLE IF NOT EXISTS domain (
  domain_id STRING NOT NULL
  ,domain_name STRING NOT NULL
  ,domain_concept_id BIGINT NOT NULL
)
USING DELTA
COMMENT 'OMOP CDM Domain definitions';

In [ ]:
-- CONCEPT_CLASS table
CREATE TABLE IF NOT EXISTS concept_class (
  concept_class_id STRING NOT NULL
  ,concept_class_name STRING NOT NULL
  ,concept_class_concept_id BIGINT NOT NULL
)
USING DELTA
COMMENT 'OMOP CDM Concept Class definitions';

In [ ]:
-- CONCEPT_RELATIONSHIP table
CREATE TABLE IF NOT EXISTS concept_relationship (
  concept_id_1 BIGINT NOT NULL
  ,concept_id_2 BIGINT NOT NULL
  ,relationship_id STRING NOT NULL
  ,valid_start_date DATE NOT NULL
  ,valid_end_date DATE NOT NULL
  ,invalid_reason STRING
)
USING DELTA
COMMENT 'OMOP CDM Concept Relationship - Maps between concepts';

In [ ]:
-- RELATIONSHIP table
CREATE TABLE IF NOT EXISTS relationship (
  relationship_id STRING NOT NULL
  ,relationship_name STRING NOT NULL
  ,is_hierarchical STRING NOT NULL
  ,defines_ancestry STRING NOT NULL
  ,reverse_relationship_id STRING NOT NULL
  ,relationship_concept_id BIGINT NOT NULL
)
USING DELTA
COMMENT 'OMOP CDM Relationship type definitions';

In [ ]:
-- CONCEPT_SYNONYM table
CREATE TABLE IF NOT EXISTS concept_synonym (
  concept_id BIGINT NOT NULL
  ,concept_synonym_name STRING NOT NULL
  ,language_concept_id BIGINT NOT NULL
)
USING DELTA
COMMENT 'OMOP CDM Concept Synonym';

In [ ]:
-- CONCEPT_ANCESTOR table
CREATE TABLE IF NOT EXISTS concept_ancestor (
  ancestor_concept_id BIGINT NOT NULL
  ,descendant_concept_id BIGINT NOT NULL
  ,min_levels_of_separation BIGINT NOT NULL
  ,max_levels_of_separation BIGINT NOT NULL
)
USING DELTA
COMMENT 'OMOP CDM Concept Ancestor - Hierarchical relationships';

In [ ]:
-- DRUG_STRENGTH table
CREATE TABLE IF NOT EXISTS drug_strength (
  drug_concept_id BIGINT NOT NULL
  ,ingredient_concept_id BIGINT NOT NULL
  ,amount_value DOUBLE
  ,amount_unit_concept_id BIGINT
  ,numerator_value DOUBLE
  ,numerator_unit_concept_id BIGINT
  ,denominator_value DOUBLE
  ,denominator_unit_concept_id BIGINT
  ,box_size BIGINT
  ,valid_start_date DATE NOT NULL
  ,valid_end_date DATE NOT NULL
  ,invalid_reason STRING
)
USING DELTA
COMMENT 'OMOP CDM Drug Strength';

In [ ]:
-- SOURCE_TO_CONCEPT_MAP table
CREATE TABLE IF NOT EXISTS source_to_concept_map (
  source_code STRING NOT NULL
  ,source_concept_id BIGINT NOT NULL
  ,source_vocabulary_id STRING NOT NULL
  ,source_code_description STRING
  ,target_concept_id BIGINT NOT NULL
  ,target_vocabulary_id STRING NOT NULL
  ,valid_start_date DATE NOT NULL
  ,valid_end_date DATE NOT NULL
  ,invalid_reason STRING
)
USING DELTA
COMMENT 'OMOP CDM Source to Concept Map';

## Load Vocabularies from S3

Load vocabularies from the public HLS S3 bucket or your own vocabulary path.

**Pattern from databricks-industry-solutions/omop-cdm:**

In [ ]:
%python
from pyspark.sql.functions import to_date, col

# Configure vocabulary source path
vocab_s3_path = spark.sql("SELECT vocab_path").collect()[0][0]
print(f"Loading vocabularies from: {vocab_s3_path}")

# List of vocabulary tables to load
vocab_tables = [
    'CONCEPT',
    'VOCABULARY', 
    'CONCEPT_ANCESTOR',
    'CONCEPT_RELATIONSHIP',
    'RELATIONSHIP',
    'CONCEPT_SYNONYM',
    'DOMAIN',
    'CONCEPT_CLASS',
    'DRUG_STRENGTH'
]

# Tables with date columns that need conversion
date_tables = ['CONCEPT', 'CONCEPT_RELATIONSHIP', 'DRUG_STRENGTH']

def load_vocab_table(table_name):
    """Load vocabulary table from CSV (supports .csv or .csv.gz)"""
    try:
        # Try gzipped first, then regular CSV
        try:
            df = spark.read.csv(
                f'{vocab_s3_path}/{table_name}.csv.gz',
                inferSchema=True,
                header=True,
                dateFormat="yyyy-MM-dd"
            )
        except:
            df = spark.read.csv(
                f'{vocab_s3_path}/{table_name}.csv',
                inferSchema=True, 
                header=True,
                dateFormat="yyyy-MM-dd"
            )
        
        # Convert date columns if needed
        if table_name in date_tables:
            if 'valid_start_date' in df.columns:
                df = df.withColumn('valid_start_date', to_date(col('valid_start_date')))
            if 'valid_end_date' in df.columns:
                df = df.withColumn('valid_end_date', to_date(col('valid_end_date')))
        
        # Write to Delta table
        df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(table_name.lower())
        
        count = df.count()
        print(f"✓ {table_name}: {count:,} records loaded")
        return count
    except Exception as e:
        print(f"✗ {table_name}: {str(e)}")
        return 0

# Uncomment to load vocabularies from S3
# for table in vocab_tables:
#     load_vocab_table(table)

print("\n" + "="*60)
print("To load full OMOP vocabularies, uncomment the loop above.")
print(f"Source: {vocab_s3_path}")
print("="*60)

## Seed Essential Concepts

Pre-populate commonly used concepts for FHIR-to-OMOP mapping.

In [ ]:
-- Insert essential concepts used in FHIR-to-OMOP mapping
MERGE INTO concept AS target
USING (
  SELECT * FROM VALUES
    -- Gender concepts
    (8507, 'MALE', 'Gender', 'Gender', 'Gender', 'S', 'M', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (8532, 'FEMALE', 'Gender', 'Gender', 'Gender', 'S', 'F', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (8551, 'UNKNOWN', 'Gender', 'Gender', 'Gender', NULL, 'U', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Visit concepts
    (9201, 'Inpatient Visit', 'Visit', 'Visit', 'Visit', 'S', 'IP', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (9202, 'Outpatient Visit', 'Visit', 'Visit', 'Visit', 'S', 'OP', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (9203, 'Emergency Room Visit', 'Visit', 'Visit', 'Visit', 'S', 'ER', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (581476, 'Home Visit', 'Visit', 'Visit', 'Visit', 'S', 'HH', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (5083, 'Telehealth', 'Visit', 'Visit', 'Visit', 'S', 'VR', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (42898160, 'Long Term Care Visit', 'Visit', 'Visit', 'Visit', 'S', 'LTCF', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (262, 'Emergency Room and Inpatient Visit', 'Visit', 'Visit', 'Visit', 'S', 'ERIP', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Type concepts
    (32817, 'EHR', 'Type Concept', 'Type Concept', 'Type Concept', 'S', 'EHR', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (32856, 'Lab', 'Type Concept', 'Type Concept', 'Meas Type', 'S', 'Lab', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (38000177, 'Prescription written', 'Type Concept', 'Type Concept', 'Drug Type', 'S', 'Rx', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Operator concepts (for measurements)
    (4172703, 'Equals', 'Meas Value Operator', 'Operator', 'Qualifier Value', 'S', '=', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4171756, 'Less than', 'Meas Value Operator', 'Operator', 'Qualifier Value', 'S', '<', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4171754, 'Less than or equal to', 'Meas Value Operator', 'Operator', 'Qualifier Value', 'S', '<=', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4172704, 'Greater than', 'Meas Value Operator', 'Operator', 'Qualifier Value', 'S', '>', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4171755, 'Greater than or equal to', 'Meas Value Operator', 'Operator', 'Qualifier Value', 'S', '>=', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Condition status concepts
    (32902, 'Active', 'Condition Status', 'Condition Status', 'Condition Status', 'S', 'active', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (32906, 'Resolved', 'Condition Status', 'Condition Status', 'Condition Status', 'S', 'resolved', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Race concepts (OMB categories)
    (8516, 'Black or African American', 'Race', 'Race', 'Race', 'S', '2054-5', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (8515, 'Asian', 'Race', 'Race', 'Race', 'S', '2028-9', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (8527, 'White', 'Race', 'Race', 'Race', 'S', '2106-3', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (8657, 'American Indian or Alaska Native', 'Race', 'Race', 'Race', 'S', '1002-5', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (8557, 'Native Hawaiian or Other Pacific Islander', 'Race', 'Race', 'Race', 'S', '2076-8', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Ethnicity concepts
    (38003563, 'Hispanic or Latino', 'Ethnicity', 'Ethnicity', 'Ethnicity', 'S', '2135-2', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (38003564, 'Not Hispanic or Latino', 'Ethnicity', 'Ethnicity', 'Ethnicity', 'S', '2186-5', DATE'1970-01-01', DATE'2099-12-31', NULL)
    
  AS source(concept_id, concept_name, domain_id, vocabulary_id, concept_class_id, standard_concept, concept_code, valid_start_date, valid_end_date, invalid_reason)
) AS source
ON target.concept_id = source.concept_id
WHEN NOT MATCHED THEN INSERT *;

## Pediatric-Specific Concepts

Add concepts commonly used in pediatric care including:
- Growth measurements (weight-for-age, height-for-age, BMI-for-age percentiles)
- Developmental milestones and assessments
- Pediatric vital signs
- Common pediatric diagnoses
- Vaccination concepts

In [ ]:
-- Pediatric-specific measurement concepts (LOINC codes)
MERGE INTO concept AS target
USING (
  SELECT * FROM VALUES
    -- Growth Charts (WHO/CDC)
    (3023540, 'Body height', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '8302-2', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (3013762, 'Body weight', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '29463-7', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (3038553, 'Body mass index', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '39156-5', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (3036277, 'Head circumference', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '9843-4', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (40762636, 'Weight-for-length percentile', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '77606-2', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (40762634, 'Height-for-age percentile', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '59574-4', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (40762635, 'Weight-for-age percentile', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '59575-1', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (40762637, 'BMI-for-age percentile', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '59576-9', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (40762633, 'Head circumference-for-age percentile', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '59578-5', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Pediatric Vital Signs
    (3004249, 'Heart rate', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '8867-4', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (3024171, 'Respiratory rate', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '9279-1', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (3020891, 'Body temperature', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '8310-5', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (40762352, 'Oxygen saturation', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '59408-5', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (3012888, 'Systolic blood pressure', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '8480-6', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (3018586, 'Diastolic blood pressure', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '8462-4', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Developmental Assessments  
    (40762499, 'Denver II Developmental Screening', 'Measurement', 'LOINC', 'Survey', 'S', '62375-3', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (40769290, 'Ages and Stages Questionnaire 3', 'Measurement', 'LOINC', 'Survey', 'S', '62378-7', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (40766721, 'PEDS Developmental Milestones', 'Measurement', 'LOINC', 'Survey', 'S', '57055-6', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (40768653, 'M-CHAT-R score', 'Measurement', 'LOINC', 'Survey', 'S', '62385-2', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (44804018, 'Pediatric Quality of Life Inventory', 'Measurement', 'LOINC', 'Survey', 'S', '71946-2', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Neonatal Measurements
    (3016335, 'Birth weight', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '8339-4', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (3015149, 'Birth length', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '89269-5', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (3004047, 'APGAR score 1 minute', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '9272-6', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (3027970, 'APGAR score 5 minute', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '9274-2', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (40766756, 'APGAR score 10 minute', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '9271-8', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (3022611, 'Gestational age at birth', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '76516-4', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (40766757, 'Birth head circumference', 'Measurement', 'LOINC', 'Clinical Observation', 'S', '11885-1', DATE'1970-01-01', DATE'2099-12-31', NULL)
    
  AS source(concept_id, concept_name, domain_id, vocabulary_id, concept_class_id, standard_concept, concept_code, valid_start_date, valid_end_date, invalid_reason)
) AS source
ON target.concept_id = source.concept_id
WHEN NOT MATCHED THEN INSERT *;

In [ ]:
-- Common pediatric condition concepts (SNOMED)
MERGE INTO concept AS target
USING (
  SELECT * FROM VALUES
    -- Common Pediatric Respiratory Conditions
    (4224709, 'Acute otitis media', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '65363002', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4170770, 'Asthma', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '195967001', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (256451, 'Bronchiolitis', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '4120002', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (25297, 'Croup', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '71186008', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4195285, 'Respiratory syncytial virus infection', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '55735004', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Developmental and Behavioral Conditions
    (4195694, 'Attention deficit hyperactivity disorder', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '406506008', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4105913, 'Autism spectrum disorder', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '35919005', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4098879, 'Developmental delay', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '248290002', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (440389, 'Speech and language disorder', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '29164008', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4034295, 'Failure to thrive', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '36813001', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Neonatal Conditions
    (4229440, 'Jaundice of newborn', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '387712008', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4029488, 'Premature birth', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '282020008', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4160552, 'Neonatal respiratory distress syndrome', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '46177005', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4082919, 'Low birth weight', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '276610007', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- GI Conditions
    (4218447, 'Gastroesophageal reflux disease', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '235595009', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (196152, 'Acute gastroenteritis', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '25374005', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4023572, 'Constipation', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '14760008', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Neurological Conditions
    (4103224, 'Febrile seizure', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '41497008', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (380378, 'Epilepsy', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '84757009', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4134440, 'Cerebral palsy', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '128188000', DATE'1970-01-01', DATE'2099-12-31', NULL),
    
    -- Allergies and Immune Conditions
    (4299535, 'Food allergy', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '414285001', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (254761, 'Atopic dermatitis', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '24079001', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (4116138, 'Allergic rhinitis', 'Condition', 'SNOMED', 'Clinical Finding', 'S', '61582004', DATE'1970-01-01', DATE'2099-12-31', NULL)
    
  AS source(concept_id, concept_name, domain_id, vocabulary_id, concept_class_id, standard_concept, concept_code, valid_start_date, valid_end_date, invalid_reason)
) AS source
ON target.concept_id = source.concept_id
WHEN NOT MATCHED THEN INSERT *;

In [ ]:
-- Pediatric vaccination concepts (CVX)
MERGE INTO concept AS target
USING (
  SELECT * FROM VALUES
    -- Pediatric Vaccines (CDC Immunization Schedule)
    (46275076, 'DTaP vaccine', 'Drug', 'CVX', 'CVX', 'S', '20', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275080, 'Hepatitis B vaccine, pediatric', 'Drug', 'CVX', 'CVX', 'S', '08', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275064, 'Haemophilus influenzae type b vaccine', 'Drug', 'CVX', 'CVX', 'S', '17', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275099, 'Inactivated polio vaccine', 'Drug', 'CVX', 'CVX', 'S', '10', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275069, 'MMR vaccine', 'Drug', 'CVX', 'CVX', 'S', '03', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275125, 'Varicella vaccine', 'Drug', 'CVX', 'CVX', 'S', '21', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275094, 'Pneumococcal conjugate PCV13', 'Drug', 'CVX', 'CVX', 'S', '133', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275106, 'Rotavirus vaccine, pentavalent', 'Drug', 'CVX', 'CVX', 'S', '116', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275081, 'Hepatitis A vaccine, pediatric', 'Drug', 'CVX', 'CVX', 'S', '83', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275098, 'Influenza vaccine, injectable', 'Drug', 'CVX', 'CVX', 'S', '141', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275089, 'Meningococcal MCV4', 'Drug', 'CVX', 'CVX', 'S', '136', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275113, 'Tdap vaccine', 'Drug', 'CVX', 'CVX', 'S', '115', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275086, 'HPV vaccine, 9-valent', 'Drug', 'CVX', 'CVX', 'S', '165', DATE'1970-01-01', DATE'2099-12-31', NULL),
    (46275117, 'COVID-19 vaccine, pediatric', 'Drug', 'CVX', 'CVX', 'S', '218', DATE'1970-01-01', DATE'2099-12-31', NULL)
    
  AS source(concept_id, concept_name, domain_id, vocabulary_id, concept_class_id, standard_concept, concept_code, valid_start_date, valid_end_date, invalid_reason)
) AS source
ON target.concept_id = source.concept_id
WHEN NOT MATCHED THEN INSERT *;

## Create Vocabulary Mapping Views

These views simplify mapping from source codes to standard concepts.

**Pattern from databricks-industry-solutions/omop-cdm:**

In [ ]:
-- Source to Standard vocabulary mapping view
CREATE OR REPLACE VIEW source_to_standard_vocab_map AS
SELECT
  c1.concept_code AS source_code
  ,c1.concept_id AS source_concept_id
  ,c1.concept_name AS source_concept_name
  ,c1.vocabulary_id AS source_vocabulary_id
  ,c1.domain_id AS source_domain_id
  ,c2.concept_id AS target_concept_id
  ,c2.concept_name AS target_concept_name
  ,c2.vocabulary_id AS target_vocabulary_id
  ,c2.domain_id AS target_domain_id
  ,c2.concept_class_id AS target_concept_class_id
  ,c2.standard_concept AS target_standard_concept
FROM concept c1
JOIN concept_relationship cr 
  ON c1.concept_id = cr.concept_id_1
  AND cr.relationship_id = 'Maps to'
  AND cr.invalid_reason IS NULL
JOIN concept c2
  ON cr.concept_id_2 = c2.concept_id
  AND c2.standard_concept = 'S'
  AND c2.invalid_reason IS NULL
WHERE c1.invalid_reason IS NULL

UNION ALL

-- Include direct source-to-concept mappings
SELECT
  s.source_code
  ,s.source_concept_id
  ,c1.concept_name AS source_concept_name
  ,s.source_vocabulary_id
  ,c1.domain_id AS source_domain_id
  ,s.target_concept_id
  ,c2.concept_name AS target_concept_name
  ,s.target_vocabulary_id
  ,c2.domain_id AS target_domain_id
  ,c2.concept_class_id AS target_concept_class_id
  ,c2.standard_concept AS target_standard_concept
FROM source_to_concept_map s
LEFT JOIN concept c1 ON s.source_concept_id = c1.concept_id
JOIN concept c2 ON s.target_concept_id = c2.concept_id
WHERE s.invalid_reason IS NULL;

In [ ]:
-- Source to Source vocabulary mapping view (for retaining source codes)
CREATE OR REPLACE VIEW source_to_source_vocab_map AS
SELECT
  c.concept_code AS source_code
  ,c.concept_id AS source_concept_id
  ,c.concept_name AS source_concept_name
  ,c.vocabulary_id AS source_vocabulary_id
  ,c.domain_id AS source_domain_id
  ,c.concept_class_id AS source_concept_class_id
  ,c.concept_id AS target_concept_id
  ,c.concept_name AS target_concept_name
  ,c.vocabulary_id AS target_vocabulary_id
  ,c.domain_id AS target_domain_id
  ,c.concept_class_id AS target_concept_class_id
FROM concept c
WHERE c.invalid_reason IS NULL

UNION ALL

SELECT
  s.source_code
  ,s.source_concept_id
  ,c.concept_name AS source_concept_name
  ,s.source_vocabulary_id
  ,c.domain_id AS source_domain_id
  ,c.concept_class_id AS source_concept_class_id
  ,s.source_concept_id AS target_concept_id
  ,c.concept_name AS target_concept_name
  ,s.source_vocabulary_id AS target_vocabulary_id
  ,c.domain_id AS target_domain_id
  ,c.concept_class_id AS target_concept_class_id
FROM source_to_concept_map s
LEFT JOIN concept c ON s.source_concept_id = c.concept_id
WHERE s.invalid_reason IS NULL;

In [ ]:
-- Verify concept counts by domain
SELECT 
  domain_id,
  COUNT(*) AS concept_count
FROM concept
GROUP BY domain_id
ORDER BY concept_count DESC;

In [ ]:
-- Show pediatric-specific concepts
SELECT 
  concept_id,
  concept_name,
  domain_id,
  vocabulary_id,
  concept_code
FROM concept
WHERE concept_name ILIKE '%pediatric%'
   OR concept_name ILIKE '%birth%'
   OR concept_name ILIKE '%percentile%'
   OR concept_name ILIKE '%apgar%'
   OR concept_name ILIKE '%developmental%'
   OR concept_name ILIKE '%neonatal%'
   OR concept_name ILIKE '%infant%'
   OR concept_name ILIKE '%vaccine%'
ORDER BY domain_id, concept_name
LIMIT 100;